# DoRA+ — Full 10k Dataset Fine-Tuning

This notebook runs the full-dataset experiment: **DoRA+**, **Qwen3-0.6B**, **Qwen2.5-0.5B**, **Smollm2-360m** and configuration **D** (full attention + MLP, rank 32).

The runtime-only configuration below is based on config D, with batch size **16**, gradient accumulation **1**, effective batch **16**, learning rate **5e-5**, and **4 epochs**. It does not modify shared fine-tuning files; this notebook stores its data, checkpoints, adapters, and reports in `full_dataset_finetune/`.

## 1. Clone repository

In [1]:
import os

REPO_URL = "https://github.com/kon172verma/intent-classifier.git"
REPO_DIR = "/content/intent-classifier"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull")

print(f"Repo at: {REPO_DIR}")

Repo at: /content/intent-classifier


## 2. Install dependencies

In [2]:
%pip install -q \
    torch \
    "torchao>=0.16.0" \
    transformers \
    accelerate \
    "peft>=0.14.0" \
    trl \
    datasets \
    bitsandbytes \
    huggingface_hub \
    python-dotenv \
    sentencepiece \
    protobuf

print("Dependencies installed.")

Dependencies installed.


## 3. Hugging Face authentication

Store your `HF_TOKEN` in **Colab Secrets** (key icon in the left sidebar).
Required for gated models (`llama3.2-1b`) and pushing adapters.

In [3]:
import os

try:
    from google.colab import userdata  # type: ignore
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("HF_TOKEN loaded from Colab Secrets.")
    else:
        print("WARNING: HF_TOKEN secret is empty.")
except Exception as e:
    print(f"Not running in Colab or secret missing: {e}")

HF_TOKEN loaded from Colab Secrets.


## 4. GPU environment check

In [4]:
import subprocess
import platform
import torch

print(f"Python  : {platform.python_version()}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
print(f"Device  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

Python  : 3.13.15
PyTorch : 2.11.0+cu128
CUDA    : 12.8
Device  : NVIDIA L4
VRAM    : 23.7 GB


## 5. Set the 10k runtime configuration

This changes only the active Colab kernel's in-memory copy of config D. The repository's source files remain untouched.

In [5]:
import runpy
import sys
from pathlib import Path

SRC_DIR = Path(REPO_DIR) / "finetune_DoRAplus" / "src"
EXPERIMENT_DIR = Path(REPO_DIR) / "full_dataset_finetune"
DATA_DIR = EXPERIMENT_DIR / "data"
ADAPTER_DIR = EXPERIMENT_DIR / "adapters"
CHECKPOINT_DIR = EXPERIMENT_DIR / "checkpoints"
TRAIN_REPORT_DIR = EXPERIMENT_DIR / "reports_training"
VAL_REPORT_DIR = EXPERIMENT_DIR / "reports_validation"
TEST_REPORT_DIR = EXPERIMENT_DIR / "reports_test"
MODEL = "smollm2-360m"
CONFIG = "D"
DATASET_SIZE = "10k"
DEVICE = "cuda"
PER_DEVICE_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 1

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from finetune_lib import LORA_CONFIGS, LORAPLUS_CONFIGS

runtime_config = {
    **LORA_CONFIGS[CONFIG],
    "per_device_train_batch_size": PER_DEVICE_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
}
LORA_CONFIGS[CONFIG] = runtime_config
LORAPLUS_CONFIGS[CONFIG] = {**LORAPLUS_CONFIGS[CONFIG], **runtime_config}

print(f"Model: {MODEL}")
print(f"Config: {CONFIG} (same adapter scope as repository config D)")
print(f"Per-device batch: {PER_DEVICE_BATCH_SIZE}")
print(f"Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
print(f"Effective batch: {PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Learning rate: {runtime_config['learning_rate']:.1e}")
print(f"Epochs: {runtime_config['num_train_epochs']}")

# Leave both as None after running section 8: the newest local registry entry is used.
# To evaluate a previously published adapter in a fresh Colab session, set both
# values from its HF subfolder, for example version="v2.0" and
# timestamp="20260827-213047".
ADAPTER_VERSION: str | None = None
ADAPTER_TIMESTAMP: str | None = None

def hub_adapter_args() -> list[str]:
    if (ADAPTER_VERSION is None) != (ADAPTER_TIMESTAMP is None):
        raise ValueError("Set both ADAPTER_VERSION and ADAPTER_TIMESTAMP, or neither.")
    if ADAPTER_VERSION is None:
        return []
    return ["--version", ADAPTER_VERSION, "--timestamp", ADAPTER_TIMESTAMP]

def run_repo_entrypoint(script_name: str, arguments: list[str]) -> None:
    previous_argv = sys.argv
    try:
        sys.argv = [str(SRC_DIR / script_name), *arguments]
        runpy.run_path(str(SRC_DIR / script_name), run_name="__main__")
    finally:
        sys.argv = previous_argv

Model: smollm2-360m
Config: D (same adapter scope as repository config D)
Per-device batch: 16
Gradient accumulation: 1
Effective batch: 16
Learning rate: 5.0e-05
Epochs: 4


## 6. Prepare the 10k 80/10/10 data split

In [6]:
import subprocess

prepare_cmd = [
    sys.executable, "-u", str(SRC_DIR / "prepare_doraplus_data.py"),
    "--dataset-size", DATASET_SIZE,
    "--out-dir", str(DATA_DIR),
]
subprocess.run(prepare_cmd, cwd=REPO_DIR, check=True)

CompletedProcess(args=['/usr/bin/python3', '-u', '/content/intent-classifier/finetune_DoRAplus/src/prepare_doraplus_data.py', '--dataset-size', '10k', '--out-dir', '/content/intent-classifier/finetune_DoRAplus/data'], returncode=0)

## 7. Optional 10-step smoke test

Run this on a fresh Colab runtime before full training. It uses the same runtime-only configuration and does not upload an adapter.

In [7]:
RUN_SMOKE_TEST = True

if RUN_SMOKE_TEST:
    run_repo_entrypoint(
        "doraplus_train.py",
        [
            "--model", MODEL,
            "--lora-config", CONFIG,
            "--dataset-size", DATASET_SIZE,
            "--device", DEVICE,
            "--data-dir", str(DATA_DIR),
            "--adapter-dir", str(ADAPTER_DIR),
            "--ckpt-dir", str(CHECKPOINT_DIR),
            "--report-dir", str(TRAIN_REPORT_DIR),
            "--gradient-checkpointing",
            "--smoke-test",
            "--no-push",
        ],
    )


  DoRA+ Training — smollm2-360m_D_10k
  Model        : HuggingFaceTB/SmolLM2-360M-Instruct
  LoRA config  : D — Heavy — full attention + MLP, rank 32
  Dataset      : 10k
  Device       : cuda
  Adapter dest : /content/intent-classifier/finetune_DoRAplus/adapters/smollm2-360m_D_10k
  HF repo      : kon172verma/intent-classifier-experiments/v2.1/smollm2-360m_DoRA+_D_10k_<timestamp>
  Mode         : SMOKE TEST (10 steps only)

  Train : 8000 examples
  Val   : 1000 examples

  Loading tokenizer: HuggingFaceTB/SmolLM2-360M-Instruct


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Dropping fully masked examples from train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

  Step 0 — train_loss=2.2741  val_loss=2.024263381958008  train_acc=0.1  val_acc=0.11

  Starting training (smoke-test: 10 steps)...



Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
10,1.057880,0.806090,0.854734,62638.000000,0.731333



  [Accuracy] step=10  train=0.3000  val=0.2100

  Training complete in 174.8s  |  Peak VRAM: 3846 MB

  TRAINING COMPLETE — smollm2-360m_D_10k
  Final train loss : 1.0579
  Final val loss   : 0.8061
  Final val acc    : 0.2100
  Final train acc  : 0.3000
  Training time    : 174.8s
  Peak VRAM        : 3846 MB
  Training report  : /content/intent-classifier/finetune_DoRAplus/reports_training/smollm2-360m_D_10k_20260828_142053.json



## 8. Train the adapter

This runs the full 10k experiment and uploads the adapter and report to Hugging Face.

In [8]:
run_repo_entrypoint(
    "doraplus_train.py",
    [
        "--model", MODEL,
        "--lora-config", CONFIG,
        "--dataset-size", DATASET_SIZE,
        "--device", DEVICE,
        "--data-dir", str(DATA_DIR),
        "--adapter-dir", str(ADAPTER_DIR),
        "--ckpt-dir", str(CHECKPOINT_DIR),
        "--report-dir", str(TRAIN_REPORT_DIR),
        "--gradient-checkpointing",
    ],
)


  DoRA+ Training — smollm2-360m_D_10k
  Model        : HuggingFaceTB/SmolLM2-360M-Instruct
  LoRA config  : D — Heavy — full attention + MLP, rank 32
  Dataset      : 10k
  Device       : cuda
  Adapter dest : /content/intent-classifier/finetune_DoRAplus/adapters/smollm2-360m_D_10k
  HF repo      : kon172verma/intent-classifier-experiments/v2.1/smollm2-360m_DoRA+_D_10k_<timestamp>

  Train : 8000 examples
  Val   : 1000 examples

  Loading tokenizer: HuggingFaceTB/SmolLM2-360M-Instruct
  Loading model:     HuggingFaceTB/SmolLM2-360M-Instruct


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


  Trainable params : 17,643,520  (4.650%)
  Total params     : 379,464,640
  Grad checkpoint  : enabled

  Tokenizing datasets...

  Effective batch  : 16
  Steps / epoch    : 500
  Total steps      : 2000
  Eval every       : 250 steps
  LoRA+ optimizer  : ratio=8  lr_A=5.0e-05  lr_B=4.0e-04


Truncating train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

...


  [Accuracy] step=0  train=0.1000  val=0.1100


Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
No log,2.024263,0,1.599411,0.000000,0.628000


  Step 0 — train_loss=2.2741  val_loss=2.024263381958008  train_acc=0.1  val_acc=0.11

  Starting training...



Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
250,0.430864,0.069365,0.057978,1725154.000000,0.983000
500,0.033405,0.006321,0.009823,3453784.000000,0.998667
750,0.006943,0.002303,0.011574,5177267.000000,0.999667
1000,0.002432,0.000158,0.001667,6907568.000000,1.000000
1250,0.000305,0.000439,0.001179,8631989.000000,0.999667
1500,0.000946,0.000206,0.001155,10361352.000000,1.000000



  [Accuracy] step=250  train=0.9200  val=0.9700

  [Accuracy] step=500  train=0.9800  val=0.9800

  [Accuracy] step=750  train=0.9800  val=1.0000

  [Accuracy] step=1000  train=1.0000  val=1.0000

  [Accuracy] step=1250  train=1.0000  val=0.9900

  [Accuracy] step=1500  train=1.0000  val=1.0000

  Training complete in 9644.1s  |  Peak VRAM: 3856 MB

  Saving adapter locally → /content/intent-classifier/finetune_DoRAplus/adapters/smollm2-360m_D_10k
  Pushing adapter to HF → kon172verma/intent-classifier-experiments/v2.1/smollm2-360m_DoRA+_D_10k_20260828-170431
  Adapter pushed successfully.
  Report pushed  : kon172verma/intent-classifier-experiments/reports/v2.1/doraplus/reports_training/smollm2-360m_D_10k_20260828_170437.json

  TRAINING COMPLETE — smollm2-360m_D_10k
  Final train loss : 0.0791
  Final val loss   : 0.0002
  Final val acc    : 1.0000
  Final train acc  : 1.0000
  Training time    : 9644.1s
  Peak VRAM        : 3856 MB
  Training report  : /content/intent-classifier/fi

## 9. Validate the published adapter

The training cell saves the final adapter locally and publishes it to the Hugging Face experiments repository. This evaluation loads that published adapter and uploads its validation report to the same repository.

In [9]:
run_repo_entrypoint(
    "doraplus_validate.py",
    [
        "--model", MODEL,
        "--lora-config", CONFIG,
        "--dataset-size", DATASET_SIZE,
        "--split", "val",
        "--device", DEVICE,
        "--data-dir", str(DATA_DIR),
        "--adapter-dir", str(ADAPTER_DIR),
        "--val-report-dir", str(VAL_REPORT_DIR),
        "--test-report-dir", str(TEST_REPORT_DIR),
        *hub_adapter_args(),
    ],
)


  DoRA+ Evaluation — smollm2-360m_D_10k
  Split    : val  (1000 examples)
  Source   : kon172verma/intent-classifier-experiments/v2.1/smollm2-360m_DoRA+_D_10k_20260828-170431
  Device   : cuda

  Loading base model: HuggingFaceTB/SmolLM2-360M-Instruct


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  Loading adapter from HF: kon172verma/intent-classifier-experiments/v2.1/smollm2-360m_DoRA+_D_10k_20260828-170431


adapter_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

v2.1/smollm2-360m_DoRA+_D_10k_20260828-1(…): reconstructing file:   0%|          |  0.00B / 70.7MB            

v2.1/smollm2-360m_DoRA+_D_10k_20260828-1(…): downloading bytes:           |  0.00B            

  [  20/1000]  running_acc=1.000
  [  40/1000]  running_acc=1.000
  [  60/1000]  running_acc=1.000
  [  80/1000]  running_acc=1.000
  [ 100/1000]  running_acc=1.000
  [ 120/1000]  running_acc=1.000
  [ 140/1000]  running_acc=1.000
  [ 160/1000]  running_acc=1.000
  [ 180/1000]  running_acc=1.000
  [ 200/1000]  running_acc=1.000
  [ 220/1000]  running_acc=1.000
  [ 240/1000]  running_acc=1.000
  [ 260/1000]  running_acc=1.000
  [ 280/1000]  running_acc=1.000
  [ 300/1000]  running_acc=1.000
  [ 320/1000]  running_acc=1.000
  [ 340/1000]  running_acc=1.000
  [ 360/1000]  running_acc=1.000
  [ 380/1000]  running_acc=1.000
  [ 400/1000]  running_acc=1.000
  [ 420/1000]  running_acc=1.000
  [ 440/1000]  running_acc=1.000
  [ 460/1000]  running_acc=1.000
  [ 480/1000]  running_acc=1.000
  [ 500/1000]  running_acc=1.000
  [ 520/1000]  running_acc=1.000
  [ 540/1000]  running_acc=1.000
  [ 560/1000]  running_acc=1.000
  [ 580/1000]  running_acc=1.000
  [ 600/1000]  running_acc=1.000
  [ 620/10

## 10. Optional locked test evaluations

Set the switch to `True` only after reviewing validation results. It evaluates both the full 1,000-example test split and the 100-example `sample_0001` anchor split used by the earlier experiments. Both reports are uploaded to the Hugging Face experiments repository.

In [10]:
RUN_TEST_EVALUATION = True

if RUN_TEST_EVALUATION:
    for split in ("test", "test_anchor"):
        run_repo_entrypoint(
            "doraplus_validate.py",
            [
                "--model", MODEL,
                "--lora-config", CONFIG,
                "--dataset-size", DATASET_SIZE,
                "--split", split,
                "--device", DEVICE,
                "--data-dir", str(DATA_DIR),
                "--adapter-dir", str(ADAPTER_DIR),
                "--val-report-dir", str(VAL_REPORT_DIR),
                "--test-report-dir", str(TEST_REPORT_DIR),
                *hub_adapter_args(),
            ],
        )


  DoRA+ Evaluation — smollm2-360m_D_10k
  Split    : test  (1000 examples)
  Source   : kon172verma/intent-classifier-experiments/v2.1/smollm2-360m_DoRA+_D_10k_20260828-170431
  Device   : cuda

  Loading base model: HuggingFaceTB/SmolLM2-360M-Instruct


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  Loading adapter from HF: kon172verma/intent-classifier-experiments/v2.1/smollm2-360m_DoRA+_D_10k_20260828-170431
  [  20/1000]  running_acc=0.950
  [  40/1000]  running_acc=0.975
  [  60/1000]  running_acc=0.983
  [  80/1000]  running_acc=0.988
  [ 100/1000]  running_acc=0.980
  [ 120/1000]  running_acc=0.983
  [ 140/1000]  running_acc=0.986
  [ 160/1000]  running_acc=0.988
  [ 180/1000]  running_acc=0.989
  [ 200/1000]  running_acc=0.990
  [ 220/1000]  running_acc=0.991
  [ 240/1000]  running_acc=0.992
  [ 260/1000]  running_acc=0.992
  [ 280/1000]  running_acc=0.989
  [ 300/1000]  running_acc=0.990
  [ 320/1000]  running_acc=0.991
  [ 340/1000]  running_acc=0.991
  [ 360/1000]  running_acc=0.992
  [ 380/1000]  running_acc=0.992
  [ 400/1000]  running_acc=0.993
  [ 420/1000]  running_acc=0.993
  [ 440/1000]  running_acc=0.993
  [ 460/1000]  running_acc=0.993
  [ 480/1000]  running_acc=0.994
  [ 500/1000]  running_acc=0.994
  [ 520/1000]  running_acc=0.994
  [ 540/1000]  running_acc=

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  Loading adapter from HF: kon172verma/intent-classifier-experiments/v2.1/smollm2-360m_DoRA+_D_10k_20260828-170431
  [  20/100]  running_acc=0.950
  [  40/100]  running_acc=0.975
  [  60/100]  running_acc=0.983
  [  80/100]  running_acc=0.988
  [ 100/100]  running_acc=0.980
  Report pushed  : kon172verma/intent-classifier-experiments/reports/v2.1/doraplus/reports_test/smollm2-360m_D_10k_test_anchor_20260828_171615.json

  Accuracy   : 0.9800  (98/100)
  Avg latency: 323.6 ms
  Peak memory: 886 MB
  Report     : /content/intent-classifier/finetune_DoRAplus/reports_test/smollm2-360m_D_10k_test_anchor_20260828_171615.json


## 11. Download reports

Creates a ZIP containing every locally generated training, validation, and test report. The final adapter is already saved locally and uploaded by the training cell; reports are also uploaded when cells 9 and 10 run.

In [11]:
import shutil
from pathlib import Path

try:
    from google.colab import files  # type: ignore

    report_dirs = {
        "reports_training": TRAIN_REPORT_DIR,
        "reports_validation": VAL_REPORT_DIR,
        "reports_test": TEST_REPORT_DIR,
    }
    staging = Path("/content/_full_dataset_reports_staging")
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir()

    for name, src in report_dirs.items():
        if src.exists() and any(src.glob("*.json")):
            shutil.copytree(src, staging / name)
        else:
            print(f"Skipping {name}: no JSON reports found")

    archive = "/content/full_dataset_doraplus_10k_reports"
    shutil.make_archive(archive, "zip", staging)
    print(f"Created {archive}.zip")
    files.download(f"{archive}.zip")
except ImportError:
    print(f"Not running in Colab. Reports are in {EXPERIMENT_DIR}/reports_*")

Created /content/doraplus_10k_reports.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>